In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

# ==========================================
# Task 3.1 — Features, split and baseline
# ==========================================
df = pd.read_csv("/content/data/appointments_clean.csv")

features = ["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
            "reminder_sent", "booking_channel", "visit_type"]
X = df[features].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline_acc = (y_test == 0).mean()
print(f"X shape: {X.shape} | Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Test no-shows: {y_test.sum()} | Baseline accuracy (always Attended): {baseline_acc:.4f}")


# ==========================================
# Task 3.2 — Logistic regression
# ==========================================
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

print("\n--- Task 3.2: Logistic Regression (Test) ---")
print(f"Accuracy: {accuracy_score(y_test, lr_pred):.3f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, lr_pred))
print(f"Precision: {precision_score(y_test, lr_pred):.3f} | Recall: {recall_score(y_test, lr_pred):.3f} | "
      f"F1: {f1_score(y_test, lr_pred):.3f} | ROC-AUC: {roc_auc_score(y_test, lr_prob):.3f}")

coef_df = pd.DataFrame({"feature": X.columns, "coef": lr.coef_[0]})
print("\nCoefficients:\n", coef_df.sort_values("coef"))

idx_rem = list(X.columns).index("reminder_sent")
coef_rem = lr.coef_[0][idx_rem]
scale_rem = scaler.scale_[idx_rem]
print(f"\nreminder_sent exp(coef) [per 1 SD]: {np.exp(coef_rem):.3f}")
print(f"reminder_sent exp(coef / scale) [No->Yes]: {np.exp(coef_rem / scale_rem):.3f}")
# Explanation: The two numbers differ because the model was fitted on standardized features (SD ≈ 0.49),
# so exp(coef) reflects a 1-SD change whereas exp(coef/scale) reflects the full 0-to-1 (No to Yes) change.
# Association: In this model, holding the other features constant, appointments with a reminder have ~69% lower odds of a no-show.


# ==========================================
# Task 3.3 — Decision tree and overfitting
# ==========================================
dt_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

print("\n--- Task 3.3: Decision Trees ---")
print(f"Full tree (depth={dt_full.get_depth()}): Train={dt_full.score(X_train, y_train):.3f}, Test={dt_full.score(X_test, y_test):.3f}")
print(f"Depth-3 tree: Train={dt_3.score(X_train, y_train):.3f}, Test={dt_3.score(X_test, y_test):.3f}")
# Gap explanation: The full-depth tree achieves 1.00 training accuracy but drops to ~0.67 on test (below the 0.7525 baseline),
# showing severe overfitting to training noise, whereas max_depth=3 generalizes much better.


# ==========================================
# Task 3.4 — Random forest and comparison table
# ==========================================
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]
dt3_pred = dt_3.predict(X_test)
dt3_prob = dt_3.predict_proba(X_test)[:, 1]

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 3 RF feature importances:\n", imp.head(3))

comp = pd.DataFrame([
    {"Model": "Logistic Regression", "Accuracy": accuracy_score(y_test, lr_pred),
     "Recall": recall_score(y_test, lr_pred), "ROC-AUC": roc_auc_score(y_test, lr_prob)},
    {"Model": "Decision Tree (depth=3)", "Accuracy": accuracy_score(y_test, dt3_pred),
     "Recall": recall_score(y_test, dt3_pred), "ROC-AUC": roc_auc_score(y_test, dt3_prob)},
    {"Model": "Random Forest (100)", "Accuracy": accuracy_score(y_test, rf_pred),
     "Recall": recall_score(y_test, rf_pred), "ROC-AUC": roc_auc_score(y_test, rf_prob)},
])
print("\n--- Task 3.4: Test Comparison Table ---\n", comp.round(3))


# ==========================================
# Task 3.5 — Tune, choose model & threshold, test once
# ==========================================
# (a) GridSearchCV on RF
gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5, scoring="roc_auc"
)
gs.fit(X_train, y_train)
print(f"\n3.5(a) Tuned RF best_params_: {gs.best_params_} | best_score_ (CV AUC): {gs.best_score_:.4f}")

# (b) 5-fold CV ROC-AUC on training set only
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_dt3 = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()

print(f"3.5(b) CV ROC-AUC -> LR: {cv_lr:.4f} | DT3: {cv_dt3:.4f} | RF: {cv_rf:.4f} | Tuned RF: {gs.best_score_:.4f}")
# Winner: Logistic Regression (highest CV ROC-AUC)

# (c) Threshold selection using OOF training probabilities
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
print("\n3.5(c) Out-of-fold Threshold Comparison:")
for th in [0.5, 0.35]:
    preds = (oof >= th).astype(int)
    print(f"Threshold {th}: Precision={precision_score(y_train, preds):.3f}, "
          f"Recall={recall_score(y_train, preds):.3f}, F1={f1_score(y_train, preds):.3f}, Flagged={preds.sum()}")
# Chosen threshold: 0.35 (higher F1 and much higher recall; a missed warning costs an empty doctor slot, while an extra call costs a minute).

# (d) Final single evaluation on test set
chosen_threshold = 0.35
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flags = (probs >= chosen_threshold).astype(int)

print(f"\n3.5(d) Final Test Evaluation (LR @ {chosen_threshold}):")
print(f"Precision={precision_score(y_test, flags):.3f}, Recall={recall_score(y_test, flags):.3f}, "
      f"Flagged={flags.sum()}, ROC-AUC={roc_auc_score(y_test, probs):.3f}")

# (e) Export test_risk_scores.csv
risk_df = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": np.round(probs, 4),
    "flag": flags
})
risk_df.to_csv("test_risk_scores.csv", index=False)
print("Saved test_risk_scores.csv with", len(risk_df), "rows and", risk_df["flag"].sum(), "flagged.")
# Real-world threshold choice: A clinic would set the threshold based on how many follow-up calls
# the front desk can realistically make in an evening, weighed against the revenue/care cost of an empty slot.